# Symmetry Phase 1 - Foundations

This notebook introduces the new Phase 1 tools:
1. synthetic sequence generation from explicit probability policies,
2. co-occurrence statistics (`P_ij`, `PMI`, `M*`),
3. concept geometry and pairwise distances.

In [ ]:
import numpy as np

from boardGPT.analysis import (
    PolicyConfig,
    SyntheticWorld,
    build_cooccurrence_statistics,
    generate_synthetic_sequences,
    lattice_1d_concept,
    lattice_nd_concept,
    pairwise_distances,
)

## 1) Build a 2D latent world

In [ ]:
concept_2d = lattice_nd_concept(shape=(8, 8), periodic=(False, False), name='grid2d')
world_2d = SyntheticWorld(concept=concept_2d)

print('vocab_size:', world_2d.vocab_size)
print('latent_dim:', concept_2d.dim)
print('first coordinates:', concept_2d.coordinates[:5])

## 2) Generate sequences with a custom policy

Example policy:
- first token sampled from uniform prior,
- transition prefers nearby tokens,
- history term keeps growth near already visited tokens (hard radius).

In [ ]:
cfg = PolicyConfig(
    length=60,
    num_sequences=1000,
    seed=0,
    transition_kernel='exponential',
    transition_scale=0.2,
    transition_strength=1.0,
    history_strength=0.8,
    history_scale=0.2,
    history_hard_radius=0.35,
    epsilon_uniform=0.05,
)

dataset = generate_synthetic_sequences(world_2d, cfg)
print('sequences shape:', dataset.sequences.shape)
print('first sequence (20 tokens):', dataset.sequences[0, :20].tolist())

## 3) Compute co-occurrence statistics

In [ ]:
stats = build_cooccurrence_statistics(
    sequences=dataset.sequences,
    vocab_size=world_2d.vocab_size,
    window_size=8,
)

print('P_ij shape:', stats.p_ij.shape)
print('sum P_ij:', float(np.sum(stats.p_ij)))
print('M* min/max:', float(np.min(stats.m_star)), float(np.max(stats.m_star)))

## 4) Compare distances and co-occurrence tendency

This quick diagnostic checks that nearer tokens tend to have larger co-occurrence.

In [ ]:
dist = pairwise_distances(concept_2d)
upper = np.triu_indices(world_2d.vocab_size, k=1)
d = dist[upper]
c = stats.p_ij[upper]
corr = np.corrcoef(d, c)[0, 1]
print('corr(distance, cooccurrence):', float(corr))

## 5) 1D periodic toy world (circle intuition)

In [ ]:
concept_1d = lattice_1d_concept(num_tokens=24, periodic=True, period=2.0, name='ring1d')
dist_1d = pairwise_distances(concept_1d)
print('periodic distance between endpoints:', float(dist_1d[0, -1]))